In [1]:
# CELL 1: Importing our Essential Python Tools (Libraries)

# WHY WE USE os:
# 'os' stands for Operating System. We use it to check file paths, check if folders
# exist, and interact with the Colab file storage system.
import os

# WHY WE USE sys:
# 'sys' gives us access to system-specific parameters and functions, such as
# monitoring Python version and memory usage.
import sys

# WHY WE USE numpy:
# 'numpy' (Numerical Python) is the foundational math library. We use it to handle
# mathematical matrices, arrays, and special values like Infinity (np.inf) and NaN (Not a Number).
import numpy as np

# WHY WE USE pandas:
# 'pandas' is the most important data analysis tool in Python. It allows us to
# load CSV files into a structured table called a "DataFrame" (just like an Excel sheet).
import pandas as pd

# WHY WE USE matplotlib.pyplot and seaborn:
# These two libraries are used to draw visual charts, histograms, and heatmaps
# so we can see the distribution of network attacks visually.
import matplotlib.pyplot as plt
import seaborn as sns



In [2]:
# Step 3: Upload the Original CICIDS2017 Dataset

from google.colab import files

uploaded = files.upload()

Saving MachineLearningCSV.zip to MachineLearningCSV.zip


In [5]:

# STEP 4: Unpack (Unzip) the 8 Dataset CSV Files

# WHY WE USE zipfile:
# Python has a built-in tool called 'zipfile' to extract zipped folders,
# just like right-clicking and selecting "Extract All" on your Windows laptop.
import zipfile
import os

# Grab the zip file name you uploaded
file_name = list(uploaded.keys())[0]
print(f" Unpacking: {file_name} ... please wait 15-20 seconds...")

# WHY WE USE extractall:
# This extracts all 8 CSV files into the '/content/' folder in Colab
with zipfile.ZipFile(file_name, 'r') as zip_ref:
    zip_ref.extractall('/content/')

print(" Unzipping Complete! All 8 CSV files are now extracted and ready!")

 Unpacking: MachineLearningCSV.zip ... please wait 15-20 seconds...
 Unzipping Complete! All 8 CSV files are now extracted and ready!


In [7]:
# STEP 5: See the 8 Extracted Attack Files

# WHY WE CHECK THE FOLDER:
# The files were extracted into a folder named 'MachineLearningCVE'.
# Let's list all 8 files so we know exactly what attack traffic we have!
folder_path = '/content/MachineLearningCVE'

# List all files that end with '.csv'
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]

print(f" Found {len(csv_files)} Attack CSV Files:\n")
for index, file in enumerate(sorted(csv_files), start=1):
    print(f"{index}. {file}")

 Found 8 Attack CSV Files:

1. Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
2. Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
3. Friday-WorkingHours-Morning.pcap_ISCX.csv
4. Monday-WorkingHours.pcap_ISCX.csv
5. Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
6. Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
7. Tuesday-WorkingHours.pcap_ISCX.csv
8. Wednesday-workingHours.pcap_ISCX.csv


In [8]:
# Step 6: Loading and combining all 8 attack files
import glob

# find all csv files inside the extracted folder
file_list = glob.glob('/content/MachineLearningCVE/*.csv')

# empty list to store each dataset
df_list = []

for file in file_list:
    print("Reading file:", file)
    # read the csv file
    data = pd.read_csv(file)

    # remove any extra spaces from column names
    data.columns = data.columns.str.strip()

    # convert float64 to float32 to reduce memory usage so Colab doesn't slow down
    for col in data.select_dtypes(include=['float64']).columns:
        data[col] = data[col].astype('float32')

    df_list.append(data)

# combine all 8 files into one single dataframe
df = pd.concat(df_list, ignore_index=True)

# check the total rows and columns
print("Combined dataset shape:", df.shape)

Reading file: /content/MachineLearningCVE/Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Friday-WorkingHours-Morning.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Monday-WorkingHours.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv
Reading file: /content/MachineLearningCVE/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Combined dataset shape: (2830743, 79)


In [9]:
# Step 7: Check the distribution of attack types
print("Unique attack types in the dataset:")
print(df['Label'].value_counts())

Unique attack types in the dataset:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


 Data Cleaning & Grouping into 12 Classes

In [10]:
# Step 8: Clean infinite/missing values and group attacks into 12 classes
import numpy as np

# 1. Replace infinite values with NaN (Not a Number) so we can clean them
# In network flows, dividing by 0 duration creates 'inf' values
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 2. Drop rows that have missing or NaN values
before_clean = len(df)
df.dropna(inplace=True)
after_clean = len(df)
print("Removed rows with NaN or infinite values:", before_clean - after_clean)

# 3. Clean the labels to make exactly 12 classes
# We combine the 3 tiny web attacks into one 'Web Attack' class
# and combine 'Heartbleed' into 'Infiltration'
label_mapping = {
    'Web Attack  Brute Force': 'Web Attack',
    'Web Attack  XSS': 'Web Attack',
    'Web Attack  Sql Injection': 'Web Attack',
    'Heartbleed': 'Infiltration'
}

# apply the mapping to our Label column
df['Label'] = df['Label'].replace(label_mapping)

# 4. Check the updated 12 classes
print("\nUpdated classes and counts (Total classes:", df['Label'].nunique(), "):")
print(df['Label'].value_counts())

Removed rows with NaN or infinite values: 2867

Updated classes and counts (Total classes: 14 ):
Label
BENIGN                        2271320
DoS Hulk                       230124
PortScan                       158804
DDoS                           128025
DoS GoldenEye                   10293
FTP-Patator                      7935
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1956
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       47
Web Attack � Sql Injection         21
Name: count, dtype: int64


In [11]:
# Step 8.1: Group any label containing 'Web Attack' into a single 'Web Attack' class
df.loc[df['Label'].str.contains('Web Attack', case=False, na=False), 'Label'] = 'Web Attack'

# Check the final classes count
print("Total unique classes now:", df['Label'].nunique())
print("\nFinal 12 Classes and their counts:")
print(df['Label'].value_counts())

Total unique classes now: 12

Final 12 Classes and their counts:
Label
BENIGN              2271320
DoS Hulk             230124
PortScan             158804
DDoS                 128025
DoS GoldenEye         10293
FTP-Patator            7935
SSH-Patator            5897
DoS slowloris          5796
DoS Slowhttptest       5499
Web Attack             2180
Bot                    1956
Infiltration             47
Name: count, dtype: int64


 Drop Constant Columns and Check for Leaky Columns

In [12]:
# Step 9: Drop constant columns (columns where all values are the same)
# 1. Find columns that have only 1 unique value
constant_cols = [col for col in df.columns if col != 'Label' and df[col].nunique() == 1]
print("Number of constant columns found:", len(constant_cols))
print("Constant columns to drop:", constant_cols)

# 2. Drop them from our dataframe
df.drop(columns=constant_cols, inplace=True)

# 3. Check if there are any leakage columns like IP addresses or Timestamps
leakage_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']
found_leakage = [col for col in leakage_cols if col in df.columns]

if len(found_leakage) > 0:
    print("Dropping data leakage columns:", found_leakage)
    df.drop(columns=found_leakage, inplace=True)
else:
    print("No IP/Timestamp leakage columns present in this CSV version. Good!")

# 4. Check the final clean shape
print("\nFinal clean dataset shape:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Number of constant columns found: 8
Constant columns to drop: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
No IP/Timestamp leakage columns present in this CSV version. Good!

Final clean dataset shape:
Rows: 2827876
Columns: 71
